# Interpretabilidad: SHAP y LIME

> Aprende a explicar las predicciones de modelos de IA con SHAP y LIME,
> y a traducir esas explicaciones técnicas a lenguaje natural con Claude.

## Objetivos
- Entrenar un clasificador con scikit-learn y calcular valores SHAP
- Generar explicaciones locales con LIME
- Usar Claude para traducir explicaciones técnicas a lenguaje accesible
- Comparar SHAP vs LIME: cuándo usar cada uno

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic shap lime scikit-learn pandas matplotlib --quiet

## 2. Entrenar clasificador en dataset de aprobación de créditos

In [ ]:
import anthropic
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

# Dataset sintético de aprobación de créditos
np.random.seed(42)
n = 300
ingresos = np.random.normal(35000, 15000, n).clip(15000, 100000)
deuda_existente = np.random.normal(10000, 8000, n).clip(0, 50000)
años_empleo = np.random.exponential(5, n).clip(0, 30)
historial_pagos = np.random.choice([0, 1], n, p=[0.3, 0.7])
edad = np.random.normal(38, 12, n).clip(18, 70)

# Variable objetivo: aprobado si ingresos altos, poca deuda y buen historial
prob_aprobado = (ingresos / 100000 * 0.4 + (1 - deuda_existente / 50000) * 0.3 +
                 historial_pagos * 0.2 + años_empleo / 30 * 0.1)
aprobado = (prob_aprobado + np.random.normal(0, 0.1, n) > 0.5).astype(int)

df = pd.DataFrame({
    "ingresos_anuales": ingresos.round(0),
    "deuda_existente": deuda_existente.round(0),
    "años_empleo": años_empleo.round(1),
    "historial_pagos_ok": historial_pagos,
    "edad": edad.round(0),
    "aprobado": aprobado
})

FEATURES = ["ingresos_anuales", "deuda_existente", "años_empleo", "historial_pagos_ok", "edad"]
X = df[FEATURES]
y = df["aprobado"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)

acc = modelo_rf.score(X_test, y_test)
print(f"Modelo entrenado. Accuracy: {acc:.2%}")
print(f"Distribución: {y.value_counts().to_dict()} (0=denegado, 1=aprobado)")

## 3. Explicar con SHAP

In [ ]:
# Calcular valores SHAP para el conjunto de test
print("Calculando valores SHAP...")
explainer_shap = shap.TreeExplainer(modelo_rf)
shap_values = explainer_shap.shap_values(X_test)

# SHAP para clase positiva (aprobado=1)
shap_aprobado = shap_values[:, :, 1] if len(shap_values.shape) == 3 else shap_values[1]

# Importancia global de features según SHAP
importancia_shap = pd.DataFrame({
    "feature": FEATURES,
    "importancia_media": np.abs(shap_aprobado).mean(axis=0)
}).sort_values("importancia_media", ascending=False)

print("=== IMPORTANCIA DE FEATURES (SHAP) ===")
print(importancia_shap.to_string(index=False))

# Explicar una predicción individual
idx = 0  # Primera muestra del test
muestra = X_test.iloc[idx]
pred = modelo_rf.predict([muestra])[0]
prob = modelo_rf.predict_proba([muestra])[0][1]

print(f"\n=== EXPLICACIÓN INDIVIDUAL (muestra #{idx}) ===")
print(f"Predicción: {'APROBADO' if pred == 1 else 'DENEGADO'} (probabilidad: {prob:.1%})")
print("Contribución de cada feature:")
for feat, val, shap_val in zip(FEATURES, muestra.values, shap_aprobado[idx]):
    signo = "→ ✓ sube" if shap_val > 0 else "→ ✗ baja"
    print(f"  {feat}: {val:.1f} {signo} probabilidad en {shap_val:.3f}")

## 4. Traducir explicación a lenguaje natural con Claude

In [ ]:
def explicar_decision_en_lenguaje_natural(muestra: pd.Series, shap_vals: np.ndarray,
                                           prediccion: int, probabilidad: float) -> str:
    """Usa Claude para traducir la explicación técnica SHAP a lenguaje comprensible."""
    datos_tecnicos = ""
    for feat, val, sv in zip(FEATURES, muestra.values, shap_vals):
        impacto = "positivo" if sv > 0 else "negativo"
        datos_tecnicos += f"- {feat}: {val:.1f} (impacto SHAP {sv:+.3f}, {impacto} para aprobación)\n"

    prompt = f"""Un sistema de evaluación de créditos ha tomado esta decisión:
Decisión: {'APROBADO' if prediccion == 1 else 'DENEGADO'} (confianza: {probabilidad:.0%})

Factores del solicitante y su impacto en la decisión:
{datos_tecnicos}

Explica esta decisión en lenguaje claro para el solicitante (no técnico).
Menciona los 2-3 factores más importantes. Sé directo, empático y constructivo.
Máximo 3 párrafos."""

    return client.messages.create(
        model=MODELO, max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text

explicacion = explicar_decision_en_lenguaje_natural(
    muestra=X_test.iloc[idx],
    shap_vals=shap_aprobado[idx],
    prediccion=int(pred),
    probabilidad=float(prob)
)
print("=== EXPLICACIÓN EN LENGUAJE NATURAL (generada por Claude) ===")
print(explicacion)

## 5. LIME vs SHAP: cuándo usar cada uno

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

# LIME: explicaciones locales con modelo sustituto lineal
explainer_lime = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=FEATURES,
    class_names=["Denegado", "Aprobado"],
    mode="classification"
)

explicacion_lime = explainer_lime.explain_instance(
    data_row=X_test.iloc[idx].values,
    predict_fn=modelo_rf.predict_proba,
    num_features=5
)

print("=== EXPLICACIÓN LIME (muestra #{}) ===".format(idx))
for feat, coef in sorted(explicacion_lime.as_list(), key=lambda x: abs(x[1]), reverse=True):
    signo = "✓" if coef > 0 else "✗"
    print(f"  {signo} {feat}: {coef:+.3f}")

print("\n=== SHAP vs LIME: CUÁNDO USAR CADA UNO ===")
comparativa = pd.DataFrame([
    {"Criterio": "Tipo de modelo", "SHAP": "Mejor con árboles (TreeSHAP)", "LIME": "Cualquier modelo (caja negra)"},
    {"Criterio": "Velocidad", "SHAP": "Rápido con TreeSHAP", "LIME": "Más lento (muestrea)"},
    {"Criterio": "Consistencia", "SHAP": "Alta (determinista)", "LIME": "Varía entre ejecuciones"},
    {"Criterio": "Explicación global", "SHAP": "✓ Excelente", "LIME": "✗ Solo local"},
    {"Criterio": "Explicación local", "SHAP": "✓ Buena", "LIME": "✓ Muy buena"},
    {"Criterio": "Facilidad de uso", "SHAP": "Media", "LIME": "Alta"},
])
print(comparativa.to_string(index=False))